# Median Income
This notebook gives a very basic example of using the `miximaps`
library to load a census detail table and display a choropleth
of that data for NYC Census tracts.

In orde to run this Notebook, you will have to
1. Get your own [Census API key](https://api.census.gov/data/key_signup.html)
2. Add it to Colab or a local .env file if you are not in Colab.


## Add a key in Colab
1. Click on the "key" icon on the left
2. CLick **Add new secret**
3. For **Name** enter: `CENSUS_API_KEY`
4. For **Value** enter the key (random number) you received from the US Census in your email.

# Install libraries
"libraries" are bundles python code that add functionality to the core language.
Some packages (e.g. `math`, `random`) are part of the language; others
are written by other developers to add specific functionality.

Here are a few of the libraries we import:
- `pandas` a key data science library for working with large sets of data
- `geopandas` enables `pandas` to work with geospatial data and to produce static and dynamic maps
- `plotly` a library for making interactive graphs and charts
- `census` makes it easier for us to request data directly from the US Census

We also install and import our own library called **miximaps**. This package contains useful functions for working with geospatial data, especially for the Census and NYC. Members of our map club contribute to this library and can help shape it.

In [2]:
# install the miximaps package for our club
# this will also install some other useful package
# that are in Colab by default
!pip install miximaps -qq


In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

from miximaps import nyc,ui

import pandas as pd
import geopandas as gpd
from census import Census


# Initialize the Census
Here we load our secret API key. This code is written
so that it can work from your local computer or
from the online Colab IDE without having to be udpated.


In [2]:
api_key = ""
try:
    from google.colab import userdata
    userdata.get('CENSUS_API_KEY')
except ImportError:
    import os
    api_key = os.environ["CENSUS_API_KEY"]

year = 2023
c = Census(api_key, year=year)

# Table B19013: Median income
- use mixi maps to load median household income for our year (2023) for
  all of the census tracts in NYC 5 boroughs and the "inner" counties
- rename the columns to make them shorter
- drop all of the rows that don't have good data



## Load and prep the data

In [3]:
# create table var for median income
table = "B19013"

# load the data
df = nyc.get_tracts(c, table, year=year, region="city", cache=True  )

df.rename(columns={"median_household_income_in_the_past_12_months_in_2023_inflation_adjusted_dollars":"median_income"}, inplace=True)

# show the column names
display(df.columns)

# drop empty tracts
df = df[df.median_income > 0]
# show a "sample" of 10 rows of data
df.sample(10)


Index(['geographic_area_name', 'geography', 'median_income',
       'total_population', 'state', 'county', 'tract', 'statefp', 'countyfp',
       'geometry', 'borough'],
      dtype='object')

,geographic_area_name,geography,median_income,total_population,state,county,tract,statefp,countyfp,geometry,borough
207,Census Tract 274.01; Bronx County; New York,1400000US36005027401,104811.0,5111.0,NY,Bronx County,027401,36,005,"MULTIPOLYGON (((-73.82624 40.84299, -73.82581 ...",Bronx
1429,Census Tract 235.02; New York County; New York,1400000US36061023502,44799.0,1872.0,NY,New York County,023502,36,061,"MULTIPOLYGON (((-73.94014 40.82704, -73.93969 ...",Manhattan
651,Census Tract 326; Kings County; New York,1400000US36047032600,48238.0,10085.0,NY,Kings County,032600,36,047,"MULTIPOLYGON (((-73.99264 40.57812, -73.99156 ...",Brooklyn
314,Census Tract 409; Bronx County; New York,1400000US36005040900,65925.0,3392.0,NY,Bronx County,040900,36,005,"MULTIPOLYGON (((-73.90048 40.87495, -73.90043 ...",Bronx
206,Census Tract 273; Bronx County; New York,1400000US36005027300,54096.0,7120.0,NY,Bronx County,027300,36,005,"MULTIPOLYGON (((-73.90616 40.87324, -73.90573 ...",Bronx
1222,Census Tract 57; New York County; New York,1400000US36061005700,222237.0,2865.0,NY,New York County,005700,36,061,"MULTIPOLYGON (((-73.99488 40.72902, -73.99455 ...",Manhattan
2122,Census Tract 731; Queens County; New York,1400000US36081073100,166481.0,1832.0,NY,Queens County,073100,36,081,"MULTIPOLYGON (((-73.85074 40.71073, -73.85033 ...",Queens
1622,Census Tract 26; Queens County; New York,1400000US36081002600,130089.0,2218.0,NY,Queens County,002600,36,081,"MULTIPOLYGON (((-73.84586 40.6962, -73.84372 4...",Queens
3,Census Tract 16; Bronx County; New York,1400000US36005001600,42957.0,6011.0,NY,Bronx County,001600,36,005,"POLYGON ((-73.86153 40.81938, -73.86203 40.821...",Bronx
205,Census Tract 269; Bronx County; New York,1400000US36005026900,29140.0,4185.0,NY,Bronx County,026900,36,005,"POLYGON ((-73.91085 40.86653, -73.91021 40.867...",Bronx


In [9]:
# load nyc neighborhoods
# they have an id number as "nta202" -- the "Neighborhood Tabulation Area"
hoods = nyc.get_neighborhoods()
display(hoods.columns.to_list())
# convert both dataframes to use meters
hoods = hoods.to_crs(nyc.crs_meters)
tracts = df.to_crs(nyc.crs_meters).copy()

# both data sets don't need boro
tracts.drop(columns="borough", inplace=True)

tracts["tract_area"] = tracts.geometry.area
hoods["nta_area"] = hoods.geometry.area

len(tracts), len(hoods)
tracts.tract_area = tracts.tract_area.astype(int)
tracts

['neighborhood', 'borough', 'nta2020', 'geometry']

,geographic_area_name,geography,median_income,total_population,state,county,tract,statefp,countyfp,geometry,tract_area
1,Census Tract 2; Bronx County; New York,1400000US36005000200,121171.0,5177.0,NY,Bronx County,000200,36,005,"POLYGON ((1022549.725 235032.264, 1022232.973 ...",4823332
2,Census Tract 4; Bronx County; New York,1400000US36005000400,98242.0,6481.0,NY,Bronx County,000400,36,005,"MULTIPOLYGON (((1024242.74 236540.07, 1024178....",8346384
3,Census Tract 16; Bronx County; New York,1400000US36005001600,42957.0,6011.0,NY,Bronx County,001600,36,005,"POLYGON ((1022576.031 237829.315, 1022436.97 2...",5221245
4,Census Tract 19.01; Bronx County; New York,1400000US36005001901,67361.0,2401.0,NY,Bronx County,001901,36,005,"POLYGON ((1003368.472 233752.96, 1003598.06 23...",2206644
5,Census Tract 19.02; Bronx County; New York,1400000US36005001902,76429.0,1979.0,NY,Bronx County,001902,36,005,"MULTIPOLYGON (((1004801.452 232650.196, 100481...",5056162
...,...,...,...,...,...,...,...,...,...,...,...
2319,Census Tract 1579.01; Queens County; New York,1400000US36081157901,114219.0,5253.0,NY,Queens County,157901,36,081,"MULTIPOLYGON (((1064521.493 211892.442, 106473...",9759862
2320,Census Tract 1579.02; Queens County; New York,1400000US36081157902,109875.0,4054.0,NY,Queens County,157902,36,081,"MULTIPOLYGON (((1062495.33 210472.21, 1062725....",7777857
2321,Census Tract 1579.03; Queens County; New York,1400000US36081157903,121000.0,3746.0,NY,Queens County,157903,36,081,"MULTIPOLYGON (((1063589.512 207614.65, 1063832...",7148861
2322,Census Tract 1617; Queens County; New York,1400000US36081161700,115125.0,4786.0,NY,Queens County,161700,36,081,"MULTIPOLYGON (((1060556.562 203300.332, 106060...",7560077


In [22]:
# since the ntas and tracts aren't nested, we need to clip partial tracts
# into NTA

inter = gpd.overlay(tracts, hoods, how="intersection")
inter["inter_area"] = inter.geometry.area
inter["tract_weight"] = inter.inter_area / inter.tract_area
inter["inter_median_inc"] = inter.median_income * inter.tract_weight

nta_income = inter[["nta2020", "inter_median_inc", "tract_weight"]].groupby( "nta2020").sum().reset_index()
nta_income.inter_median_inc = (nta_income.inter_median_inc / nta_income.tract_weight).round()

nta_income.inter_median_inc.mean(), tracts.median_income.mean()

(np.float64(90434.01960784313), np.float64(88549.24178832117))

In [ ]:
# since the ntas and tracts aren't nested, we need to clip partial tracts
# into NTA

inter = gpd.overlay(tracts, hoods, how="intersection")
# calculate the area percent of each tract-nta intersection of the total tract
inter["area_weight"] = inter.area / inter.groupby("tract").area.transform("sum")

# # 3. Compute weighted median income contribution
inter["weighted_inc"] = inter["median_income"] * inter["area_weight"]

# # 4. Aggregate to NTA
nta_income = inter[["nta2020", "weighted_inc", "area_weight"]].groupby("nta2020").sum().reset_index()
nta_income["median_income"] = round(nta_income.weighted_inc / nta_income.area_weight)
nta_income
nta_income.apply(lambda g: g["weighted_inc"].sum() / g["area_weight"].sum())
    .apply(lambda g: g["weighted_inc"].sum() / g["area_weight"].sum())
    .reset_index(name="median_income")
)
# nta_income.columns = ["nta2020", "median_income"]
# nta_income.median_income = nta_income.median_income.astype(int)
nta_income.median_income.mean(), tracts.median_income.mean()

(np.float64(89897.63921568627), np.float64(88549.24178832117))

In [9]:

m = ui.base_map(hoods)
m = hoods.explore(m=m)
m = ui.make_labels(m, hoods, "neighborhood")
m

## Make a map
This "choropleth" uses shades of Purple to show differenes in median income.
Darker purple tracts have a higher median income. Note that some tracts are empty because we did not have valid data returned for thos tracts.

In [10]:
# create a new column that formats median income as a whole dollar string
df["Median Income"] = df["median_income"].apply(lambda x: f"${x:,.0f}")

# list the columns we want to show in the popup,
# in the order we want them to appear
cols =["geographic_area_name", "Median Income", "county", "state"]

# create a new column with the pop info
df["popup"] = df.apply(ui.popup(cols), axis=1)

# get a basemap using the (default) cartodb tiles
m = ui.base_map(df)

# create the map
m = df.explore(m=m, column="median_income", tooltip="Median Income",
    popup="popup", cmap="Purples", popup_kwds=dict(labels=False),
    style_kwds=dict(fillOpacity=1, opacity=1))
m.save("median-inc.html")
